## Unity Catalog Hosted SQL Functions with Databricks Agents

![UDF_SQL_UC_Function](./Assets/UDF_SQL_UC_Function.png)

### Installing Utilities and Libraries

In [0]:
%pip install databricks-langchain==0.12.1 langchain-community==0.4.1 langchain-experimental==0.4.1 unitycatalog-ai[databricks]==0.3.2 unitycatalog-langchain[databricks]==0.3.0

### Restarting the Python Kernel

In [0]:
dbutils.library.restartPython()

### Importing CSV File into Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS genai_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/genai_lab"

# List of files to download
files = ["electronics_products.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/LangChain/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Registering the CSV as a Delta Table in Unity Catalog

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/genai_lab/electronics_products.csv', format='csv', header='true', inferSchema='true')
display(df.limit(100))

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the CSV dataset as a Delta Table in the Databricks File System (DBFS)
deltaTablePath = f'/Volumes/{catalog_name}/default/genai_lab/delta/electronics_products'
df.write.format('delta').mode('overwrite').save(deltaTablePath)

# Storing the CSV dataset as a Delta Table in the Data Catalog
df.write.format('delta').saveAsTable('default.electronics_products')

### Creating the Unity Catalog Hosted Functions Client

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

### Defining the Tool Logic and Registering in Unity Catalog

In [0]:
%sql
CREATE OR REPLACE FUNCTION default.lookup_electronics_item(
  product_name STRING COMMENT 'Name of the product to look up. for instance if user query is "how much does green webcam cost", then product anem is "webcam" and not "green webcam"',
  product_colour STRING COMMENT 'Colour of the product to look up.'
)
RETURNS STRING
COMMENT 'Returns metadata about a specific product in the electronics_items dataset, including its ID, price, and description.'
RETURN SELECT CONCAT(
    'Product ID: ', productID, ', ',
    'Product Name: ', productName, ', ',
    'Product Colour: ', colour, ', ',
    'Price: ', price, ', '
  )
  FROM default.electronics_products
  WHERE LOWER(productName) = LOWER(product_name) 
    AND LOWER(colour) = LOWER(product_colour)
  LIMIT 1;

     

### Creating the Tool Object for usage within LangChain Agent

In [0]:
from databricks_langchain import UCFunctionToolkit

# Create a toolkit with the Unity Catalog function
func_name = f"{CATALOG}.{SCHEMA}.lookup_electronics_item"
toolkit = UCFunctionToolkit(function_names=[func_name])

tools = toolkit.tools

### Creating the tool-calling Agent with LangChain

In [0]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain import (
  ChatDatabricks,
  UCFunctionToolkit,
)
import mlflow

# Initialize the LLM
LLM_ENDPOINT_NAME = "databricks-claude-haiku-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.1)

# Define the prompt with agent_scratchpad placeholder
prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a helpful assistant. Make sure to use tools for additional functionality.",
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
  ]
)

# Enable automatic tracing
mlflow.langchain.autolog()

# Define the agent, specifying the tools from the toolkit above
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

### Invoking the Agent with the Agent Executor

In [0]:
agent_executor.invoke({"input": "Return details for the Red Graphics Card"})

In [0]:
agent_executor.invoke({"input": "Return details for the blue keyboard"})